In [1]:
# Functions to support MWAPI calls to wbsearchentities, CirrusSearch and generator search 
# With examples

import requests

S = requests.Session()
S.headers["User-Agent"] = "mwapi-substitute/0.5 (User: your_UA_contact)"
MW_URL = "https://www.wikidata.org/w/api.php"     # Endpoint for Wikidata
EN_MW_URL = "https://en.wikipedia.org/w/api.php"  # Endpoint for English language Wikipedia/Commons

timeout = 30
limit = 50

# (A few) Generator name to limit parameter mappings (i.e., the module prefix + "limit")
GEN_LIMIT = {
    "categorymembers": "gcmlimit", "search": "gsrlimit",
    "geosearch": "ggslimit", "exturlusage": "geulimit",
    "backlinks": "gbllimit", "embeddedin": "geilimit",
    "linkshere": "glhlimit", "transcludedin": "gtilimit",
    "prefixsearch": "gpslimit", "allpages": "gaplimit",
}


def api_get(api, params):
    """
    GET request to the specified API (either MW_URL or EN_MW_URL) with various parameters 
    and minimal timeout/error handling.
    """
    r = S.get(api, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    if "error" in data:                       # e.g. maxlag, bad params, throttling
        raise RuntimeError(data["error"])
    return data


def wbsearchentities(term, lang="en"):
    """
    wbsearchentities query (Wikidata label/alias search) for the input string, 
    'term', in English, with continuation processing

    Note that strictlanguage is not set as a parameter, so language=en includes 
    'mul' labels via the fallback chain.
    """
    out, offset = [], 0
    while True:
        r = api_get(MW_URL, {
            "action": "wbsearchentities", "search": term, "language": lang,
            "uselang": lang, "type": "item", "limit": min(limit, 50),
            "continue": offset, "format": "json", "formatversion": 2,
        })
        out += [h["id"] for h in r.get("search", [])]
        if "search-continue" not in r:
            return out
        offset = r["search-continue"]


def cirrus_search(srsearch):
    """
    CirrusSearch full text search based on the input string, srsearch 
    (using keywords), in namespace 0 (the main namespace)

    When addressing the request to the endpoint, MW_URL, namespace 0 refers to 
      Wikidata and returns QIDs.
    When addressed to EN_MW_URL, namespace 0 refers to Wikipedia and
      returns the titles of articles.
    """
    out, cont = [], {"continue": ""}   # Using empty continuation on first call
    while True:
        r = api_get(MW_URL, {
            "action": "query", "list": "search", "srsearch": srsearch,
            "srlimit": limit, "srnamespace": 0,
            "format": "json", "formatversion": 2, **cont,
        })
        out += [h["title"] for h in r["query"]["search"]]
        if "continue" not in r:
            return out
        cont = r["continue"]


def generator_to_qids(api, generator, params, max_results=100):
    """
    Generator search with cross-linking from Wikipedia/Commons results 
      to Wikidata QIDs.
    QIDs generated using pageprops.wikibase_item.
    max_results: Limit on the number of QIDs (None -> all)
    """
    limit_param = GEN_LIMIT.get(generator)
    base = {
        "action": "query", "generator": generator,
        "prop": "pageprops", "ppprop": "wikibase_item",
        "format": "json", "formatversion": 2,
    }
    if limit_param and limit_param not in params:
        base[limit_param] = min(max_results, 100) if max_results else "max"
    out, cont = [], {"continue": ""}
    while True:
        r = api_get(api, {**base, **params, **cont})
        for p in r.get("query", {}).get("pages", []):
            qid = p.get("pageprops", {}).get("wikibase_item")
            if qid:
                out.append(qid)
        if max_results and len(out) >= max_results:
            return out[:max_results]          # hard cap
        if "continue" not in r:
            return out
        cont = r["continue"]


In [2]:
# Example: Einstein in Wikidata label or alias -> QIDs
wbsearch_qids = wbsearchentities("Einstein")
print("wbsearchentities QIDs")
print(wbsearch_qids)
print()

# Example: CirrusSearch for occupation (P106) = astronaut (Q11631), AND 
#   sex/gender (P21) = female (Q6581072)
# Returns QIDs because namespace 0 and endpoint (hardcoded in the function) = WM_URL
cirrus_qids = cirrus_search("haswbstatement:P106=Q11631 haswbstatement:P21=Q6581072")
print("CirrusSearch QIDs")
print(cirrus_qids)
print()

# Example: Generator (categorymembers) search for English Wikipedia category 
#    (Nobel laureats) and then mapping to QIDs
cat_qids = generator_to_qids(EN_MW_URL, "categorymembers",
    {"gcmtitle": "Category:Nobel laureates in Physics", "gcmtype": "page"},
)
print("Category QIDs") 
print(cat_qids)
print()

# Example: Generator (exturlusage) search for items whose English wiki article links 
#    to nature.com (returns articles with an external link to the nature.com domain.)
# And then maps articles to QIDs
cited_qids = generator_to_qids(EN_MW_URL, "exturlusage",
    {"geuquery": "nature.com", "geuprotocol": "https", "geunamespace": 0},
)
print("Citation QIDs") 
print(cited_qids)
print()


# Example: Generator (search) for articles citing nature (Same goal as above, 
#   different mechanism)
# Returns articles whose wikitext contains the DOI string 10.1038/nature
# And then maps articles to QIDs
doi_qids = generator_to_qids(EN_MW_URL, "search",
    {"gsrsearch": 'insource:"10.1038/nature"', "gsrnamespace": 0},
)
print("Article QIDs")
print(doi_qids)
print()


wbsearchentities QIDs
['Q16834800', 'Q937', 'Q901448', 'Q1309274', 'Q26854017', 'Q1892', 'Q28739649', 'Q11452', 'Q2030894', 'Q146709', 'Q106846857', 'Q28843527', 'Q23691666', 'Q28344686', 'Q1060565', 'Q43514', 'Q27492829', 'Q11282510', 'Q8772688', 'Q52795967', 'Q18615182', 'Q113734069', 'Q766418', 'Q83654806', 'Q3049464', 'Q3292175', 'Q3720722', 'Q5349785', 'Q5349786', 'Q44652763', 'Q19971626', 'Q109416051', 'Q19115891', 'Q52856195', 'Q52792841', 'Q52791373', 'Q509660', 'Q998964', 'Q116006972', 'Q782022', 'Q1426842', 'Q60376230', 'Q19115911', 'Q35875', 'Q17712', 'Q30765887', 'Q4710134', 'Q273711', 'Q577432', 'Q214975', 'Q59151', 'Q1309294', 'Q1309281', 'Q118109752', 'Q15987049', 'Q337789', 'Q97273383', 'Q321789', 'Q57654242', 'Q77010478', 'Q67204806', 'Q65369339', 'Q122330882', 'Q117083138', 'Q111612', 'Q2291287', 'Q136164891', 'Q19115933', 'Q673253', 'Q27721667', 'Q390003', 'Q2441281', 'Q169497', 'Q138088407', 'Q1630886', 'Q1670676', 'Q131921399', 'Q131938470', 'Q130561353', 'Q1319384

In [3]:
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"   

def run(qids):
    values = " ".join(f"wd:{q}" for q in qids)
    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?item ?type ?label ?typeLabel WHERE {{
      VALUES ?item {{ {values} }}
      ?item wdt:P31 ?type .          # apply real constraints here
      OPTIONAL {{ ?type rdfs:label ?typeLabel . FILTER(LANG(?typeLabel) = "en") }}
      OPTIONAL {{ ?type rdfs:label ?typeLabel . FILTER(LANG(?typeLabel) = "mul") }}
      OPTIONAL {{ ?item rdfs:label ?label . FILTER(LANG(?label) = "en") }}
      OPTIONAL {{ ?item rdfs:label ?label . FILTER(LANG(?label) = "mul") }}
    }}"""
    return S.get(WDQS_ENDPOINT, params={"query": query},
                 headers={"Accept": "application/sparql-results+json"}).json()

# Execute the query for the first set of QIDs (wbsearchentities results)
print(run(wbsearch_qids))


{'head': {'vars': ['item', 'type', 'label', 'typeLabel']}, 'results': {'bindings': [{'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q136164891'}, 'label': {'xml:lang': 'mul', 'type': 'literal', 'value': 'Einstein'}, 'type': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q3331189'}, 'typeLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'version, edition or translation'}}, {'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q11452'}, 'label': {'xml:lang': 'en', 'type': 'literal', 'value': 'general relativity'}, 'type': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q214070'}, 'typeLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'physical law'}}, {'item': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q59151'}, 'label': {'xml:lang': 'en', 'type': 'literal', 'value': 'cosmological constant'}, 'type': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q173227'}, 'typeLabel': {'xml:lang': 'en', 'type': 'literal'

In [4]:
def pretty_print(results, maxw=60):
    """Pretty-print SPARQL JSON results (sparql-results+json) as an aligned table."""
    cols = results["head"]["vars"]
    rows = results["results"]["bindings"]

    def cell(b, v):
        if v not in b:                       # unbound (e.g. OPTIONAL var)
            return ""
        val = b[v]["value"]
        if val.startswith("http://www.wikidata.org/entity/"):
            val = val.rsplit("/", 1)[-1]     # shorten URI -> QID
        return val if len(val) <= maxw else val[:maxw - 1] + "…"

    width = {v: max([len(v)] + [len(cell(b, v)) for b in rows]) for v in cols}
    row = lambda parts: " | ".join(p.ljust(width[c]) for c, p in zip(cols, parts))
    print(row(cols))
    print("-+-".join("-" * width[c] for c in cols))
    for b in rows:
        print(row([cell(b, c) for c in cols]))
    print(f"\n{len(rows)} row(s)")

pretty_print(run(wbsearch_qids))


item       | type       | label                                                     | typeLabel                                                  
-----------+------------+-----------------------------------------------------------+------------------------------------------------------------
Q136164891 | Q3331189   | Einstein                                                  | version, edition or translation                            
Q11452     | Q214070    | general relativity                                        | physical law                                               
Q59151     | Q173227    | cosmological constant                                     | physical constant                                          
Q138088407 | Q1542966   | Einstein-Gymnasium Neuenhagen                             | gymnasium                                                  
Q135050506 | Q7889      | Einstein's Cats                                           | video game                            